# Speeding things up with Cython

pysndlib itself is compiled with Cython, and the instrument files in `pysndlib/instruments` are written in **pure-Python-mode Cython**: ordinary `.py` files that the build compiles. This notebook shows what that buys and how to apply it to your own instruments.

Docs: [building](https://testcase.github.io/pysndlib/building.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm
import pysndlib.sndlib as sndlib
import cython

SR = 44100

## Is pysndlib compiled?

In a normal install, yes -- which is why per-sample generator calls are fast:

In [ ]:
%load_ext cython

In [ ]:
import importlib, pathlib
fvm = importlib.import_module('pysndlib.instruments.fm_violin')
print('notebook python compiled:', cython.compiled)
print('instruments compiled    :', fvm.__file__.endswith(('.so', '.pyd')))

## A plain-Python instrument

The straightforward way to write an instrument -- a pure Python function. Render with it and measure:

In [ ]:
def plain_fm(beg, dur, freq, index=2.0):
    start = clm.seconds2samples(beg); length = clm.seconds2samples(dur)
    car = clm.make_oscil(freq); mod = clm.make_oscil(freq)
    amp = clm.make_env([0, 0, 5, 1, 100, 0], scaler=0.3, duration=dur)
    idx = clm.make_env([0, 0, 100, 1], scaler=clm.hz2radians(freq) * index, duration=dur)
    for i in range(start, start + length):
        clm.outa(i, clm.env(amp) * clm.oscil(car, clm.oscil(mod) * clm.env(idx)))

def render(f):
    with clm.Sound('bench.wav', channels=1, srate=SR):
        f(0.0, 1.0, 440.0)

In [ ]:
%timeit -n 3 render(plain_fm)

## The same instrument, compiled in a cell

With the `%%cython` magic, Cython compiles this cell to C. `@cython.ccall` plus typed locals removes Python overhead from the sample loop (the magic needs `pip install cython`):

In [ ]:
%%cython
import cython
import pysndlib.clm as clm

@cython.ccall
def compiled_fm(beg, dur, freq, index=2.0):
    start: cython.long = clm.seconds2samples(beg)
    length: cython.long = clm.seconds2samples(dur)
    freq: cython.double = freq
    car = clm.make_oscil(freq); mod = clm.make_oscil(freq)
    amp = clm.make_env([0, 0, 5, 1, 100, 0], scaler=0.3, duration=dur)
    idx = clm.make_env([0, 0, 100, 1], scaler=clm.hz2radians(freq) * index, duration=dur)
    i: cython.long
    for i in range(start, start + length):
        clm.outa(i, clm.env(amp) * clm.oscil(car, clm.oscil(mod) * clm.env(idx)))
    return 0.0

In [ ]:
%timeit -n 3 render(compiled_fm)

In [ ]:
# both must sound identical
with clm.Sound('bench_plain.wav', channels=1, srate=SR): plain_fm(0, 0.3, 440.0)
with clm.Sound('bench_compiled.wav', channels=1, srate=SR): compiled_fm(0, 0.3, 440.0)
a, _ = clm.file2ndarray('bench_plain.wav'); b, _ = clm.file2ndarray('bench_compiled.wav')
print('identical output:', np.allclose(a, b))

## What the shipped instruments do

The files in `pysndlib/instruments` use the same techniques plus a few more:
- typed generator locals, so calls into `clm` skip runtime type checks
- `cython.compiled` checks, so the same file runs (slower) under plain Python
- the library's own hot paths use typed memoryviews and `noexcept` wrappers

In [ ]:
src = pathlib.Path(fvm.__file__).with_name('fm_violin.py').read_text()
print(src[:1200])

## Summary

- Instruments are just functions; the sample loop is where Python overhead lives
- `%%cython` + `@cython.ccall` + typed locals speeds that loop up several-fold
- The shipped instruments show the full pattern

Next: [FM theory](06_FM_Theory.ipynb)